Mount Drive and enter the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ===== COLAB STARTUP CELL =====

from pathlib import Path
import os
import shutil
import subprocess
from datetime import datetime
import zipfile

# -------------------------
# CHANGE THESE IF NEEDED
# -------------------------
DRIVE_PROJECT = Path("/content/drive/MyDrive/CS4248_Project")  # master copy in Drive
REPO_NAME = "CS4248_Project"                                        # runtime folder name

# All persistent files will go here in THIS Google Drive account
DRIVE_ROOT = Path("/content/drive/MyDrive/CS4248_colab/preprocessing")
OUTPUT_DIR = DRIVE_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
GENERATED_DIR = OUTPUT_DIR / "generated_files"
SNAPSHOT_DIR = DRIVE_ROOT / "repo_snapshots"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"

# Fast runtime location
PROJECT_DIR = Path("/content") / REPO_NAME

# -------------------------
# SETUP FOLDERS
# -------------------------
for p in [DRIVE_ROOT, OUTPUT_DIR, TABLE_DIR, GENERATED_DIR, SNAPSHOT_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# -------------------------
# COPY PROJECT FROM DRIVE -> /content
# -------------------------
if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(f"Drive project folder not found: {DRIVE_PROJECT}")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

print(f"Copying project from Drive:\n  {DRIVE_PROJECT}\n-> {PROJECT_DIR}")
shutil.copytree(DRIVE_PROJECT, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("Working directory:", PROJECT_DIR)

# Optional: install requirements if your repo has them
if Path("requirements.txt").exists():
    print("Installing requirements.txt ...")
    subprocess.run(["pip", "install", "-r", "requirements.txt"], check=False)

# -------------------------
# HELPER FUNCTIONS
# -------------------------
def save_table(df, filename, index=False):
    path = TABLE_DIR / filename
    df.to_csv(path, index=index)
    print("Saved table ->", path)
    return path

def save_top100(df, filename):
    path = TABLE_DIR / filename
    df.head(100).to_csv(path, index=False)
    print("Saved top100 ->", path)
    return path

def copy_if_exists(rel_path):
    """
    Copy a file from the runtime repo into Drive, preserving subfolders.
    Example:
        copy_if_exists("artifacts/system_a/test_outputs.jsonl")
    """
    src = PROJECT_DIR / rel_path
    if src.exists():
        dst = GENERATED_DIR / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print("Copied ->", dst)
        return dst
    else:
        print("Missing, skipped ->", src)
        return None

def sync_repo_back_to_drive():
    """
    Overwrite the Drive project folder with the current runtime project.
    Use this if you edited code in Colab and want those edits saved to Drive.
    """
    if DRIVE_PROJECT.exists():
        shutil.rmtree(DRIVE_PROJECT)
    shutil.copytree(PROJECT_DIR, DRIVE_PROJECT)
    print("Synced runtime project back to Drive ->", DRIVE_PROJECT)
    return DRIVE_PROJECT

def backup_repo_zip(tag=None):
    """
    Zip the whole runtime repo into Drive so you can restore it later.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    name = f"{REPO_NAME}_{tag}_{timestamp}" if tag else f"{REPO_NAME}_{timestamp}"
    zip_path = SNAPSHOT_DIR / f"{name}.zip"

    skip_parts = {".git", "__pycache__", ".ipynb_checkpoints"}
    skip_suffixes = {".pyc"}

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in PROJECT_DIR.rglob("*"):
            if not path.is_file():
                continue
            if any(part in skip_parts for part in path.parts):
                continue
            if path.suffix in skip_suffixes:
                continue
            arcname = path.relative_to(PROJECT_DIR.parent)
            zf.write(path, arcname)

    print("Repo snapshot saved ->", zip_path)
    return zip_path

print("\nPersistent folders:")
print("DRIVE_PROJECT  =", DRIVE_PROJECT)
print("PROJECT_DIR    =", PROJECT_DIR)
print("OUTPUT_DIR     =", OUTPUT_DIR)
print("TABLE_DIR      =", TABLE_DIR)
print("GENERATED_DIR  =", GENERATED_DIR)
print("CHECKPOINT_DIR =", CHECKPOINT_DIR)
print("SNAPSHOT_DIR   =", SNAPSHOT_DIR)

Copying project from Drive:
  /content/drive/MyDrive/CS4248_Project-preprocessing
-> /content/CS4248_Project
Working directory: /content/CS4248_Project
Installing requirements.txt ...

Persistent folders:
DRIVE_PROJECT  = /content/drive/MyDrive/CS4248_Project-preprocessing
PROJECT_DIR    = /content/CS4248_Project
OUTPUT_DIR     = /content/drive/MyDrive/CS4248_colab/preprocessing/outputs
TABLE_DIR      = /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/tables
GENERATED_DIR  = /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/generated_files
CHECKPOINT_DIR = /content/drive/MyDrive/CS4248_colab/preprocessing/checkpoints
SNAPSHOT_DIR   = /content/drive/MyDrive/CS4248_colab/preprocessing/repo_snapshots


In [ ]:
!pwd
!ls


/content/CS4248_Project
 analysis
 artifacts
 classifiers
 data
 DATA.md
 dataset
 deep-research-report.md
 execution_roadmap.md
 experiments
 generation
 human_eval
 metrics
 notebooks
 requirements-m1.txt
 requirements.txt
 rerank
'Responsibility Assignment for a Six-Person Retrieve–Edit–Rerank Sarcasm Transfer Project.pdf'
 retrieval
 scripts
 similarity
 splits
 SYSTEM_B_MANUAL_WORKFLOW.md
 systems


Install dependencies

In [ ]:
!pip install -r requirements.txt


Check GPU

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


True
Tesla T4


Train the split-aware sarcasm detectors

In [ ]:
!python classifiers/train_classifiers.py --split topic_hard --models all


Split: standard
Models: ['logreg', 'rnn', 'transformer']
Outputs: artifacts/classifiers/<kind>/standard/

--- LogReg (TF-IDF) ---
Saved model -> /content/CS4248_Project/artifacts/classifiers/logreg/standard/model.joblib
Saved metrics -> /content/CS4248_Project/artifacts/classifiers/logreg/standard/metrics.json
{
  "dev": {
    "accuracy": 0.8557894736842105,
    "precision": 0.826417704011065,
    "recall": 0.8819188191881919,
    "f1": 0.8532666904676901,
    "roc_auc": 0.9295432499475498
  },
  "test": {
    "accuracy": 0.8411083830235005,
    "precision": 0.8155136268343816,
    "recall": 0.8606194690265486,
    "f1": 0.83745963401507,
    "roc_auc": 0.9169818766586754
  },
  "config": {
    "model_type": "logreg",
    "split": "standard",
    "text_col": "text",
    "max_features": 50000,
    "c_value": 1.0
  }
}

--- RNN (BiLSTM) ---
Epoch 1/3: 100% 357/357 [00:04<00:00, 88.14it/s] 
Epoch 1: loss=0.4782, dev_f1=0.8153
Epoch 2/3: 100% 357/357 [00:03<00:00, 111.68it/s]
Epoch 2: loss

Train System B on the author-balanced pseudo-pairs

In [ ]:
!python systems/system_b_train.py --split topic_hard --epochs 4 --batch-size 8


tokenizer_config.json: 2.54kB [00:00, 11.1MB/s]
spiece.model: 100% 792k/792k [00:01<00:00, 648kB/s] 
tokenizer.json: 2.42MB [00:00, 123MB/s]
special_tokens_map.json: 2.20kB [00:00, 12.6MB/s]
config.json: 1.40kB [00:00, 6.38MB/s]
2026-04-15 03:27:59.235729: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776223679.268849    4425 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776223679.280269    4425 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776223679.308057    4425 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776223679.308092    

Generate and evaluate System B on the held-out test split

In [ ]:
!python systems/system_b_evaluate.py --split-name test --split-strategy topic_hard --limit 100 --batch-size 16


2026-04-14 15:51:02.737452: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776181862.759057    6123 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776181862.766117    6123 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776181862.789187    6123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776181862.789215    6123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776181862.789220    6123 computation_placer.cc:177] computation placer alr

Evaluate System C without retrieval

In [ ]:
!python systems/system_c_evaluate.py --split-name test --split-strategy topic_hard --limit 100 --no-retrieval


2026-04-14 15:51:28.900295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776181888.922772    6261 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776181888.930086    6261 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776181888.954946    6261 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776181888.954976    6261 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776181888.954981    6261 computation_placer.cc:177] computation placer alr

Build the retrieval index for System C

In [ ]:
!python retrieval/build_index.py --split topic_hard


2026-04-14 15:52:24.930695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776181944.951903    6507 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776181944.960707    6507 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776181944.979607    6507 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776181944.979662    6507 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776181944.979666    6507 computation_placer.cc:177] computation placer alr

Evaluate System C with retrieval

In [ ]:
!python systems/system_c_evaluate.py --split-name test --split-strategy topic_hard --limit 100


2026-04-14 15:55:48.489648: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776182148.534189    7354 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776182148.546881    7354 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776182148.577619    7354 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776182148.577660    7354 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776182148.577665    7354 computation_placer.cc:177] computation placer alr

Generate System A outputs on the same test split

In [ ]:
import json
from pathlib import Path
import pandas as pd

from systems.system_a.system_a_template import SystemATemplate
from systems.system_b_utils import infer_direction, load_anchored_data, select_subset

system_a = SystemATemplate()

df = load_anchored_data()
test_df = select_subset(df, split_name="test", split="topic_hard").reset_index(drop=True)

rows = []
for _, row in test_df.iterrows():
    direction = infer_direction(int(row["label"]))
    rewritten = system_a.rewrite(row["text"], row.get("anchors", {}), direction)
    rows.append({
        "id": row["id"],
        "direction": direction,
        "source_text": row["text"],
        "target_text": rewritten,
    })

out_path = Path("artifacts/system_a/test_outputs.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_json(out_path, orient="records", lines=True, force_ascii=False)
print(f"Saved -> {out_path}")


Saved -> artifacts/system_a/test_outputs.jsonl


Build the overall comparison table

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

from classifiers.batch_style_probs import (
    available_detectors,
    batch_probs_for_all_detectors,
    pick_primary_detector,
)
from systems.system_a.template_utils import preserves_anchors
from systems.system_b_utils import load_anchored_data, select_subset, semantic_similarity_score

PROJECT_ROOT = Path(".").resolve()
SPLIT = "topic_hard"
SPLIT_NAME = "test"

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

a = pd.DataFrame(read_jsonl("artifacts/system_a/test_outputs.jsonl"))
b = pd.DataFrame(read_jsonl("artifacts/system_b/topic_hard/test_outputs.jsonl"))
c_no = pd.DataFrame(read_jsonl("results/system_c_outputs_topic_hard_test_no_retrieval.jsonl"))
c_yes = pd.DataFrame(read_jsonl("results/system_c_outputs_topic_hard_test.jsonl"))

gold = load_anchored_data()
gold = select_subset(gold, split_name=SPLIT_NAME, split=SPLIT)[["id", "label", "anchors"]].copy()

avail = available_detectors(PROJECT_ROOT, SPLIT)
primary = pick_primary_detector(avail)
print("Available detectors:", {k: str(v) if v else None for k, v in avail.items()})
print("Primary detector:", primary)

def enrich_outputs(df):
    merged = df.merge(gold, on="id", how="left")
    if "semantic_similarity" not in merged.columns:
        merged["semantic_similarity"] = merged.apply(
            lambda row: semantic_similarity_score(row["source_text"], row["target_text"]),
            axis=1,
        )
    if "anchors_preserved" not in merged.columns:
        merged["anchors_preserved"] = merged.apply(
            lambda row: preserves_anchors(row.get("anchors", {}), row["target_text"]),
            axis=1,
        )

    detector_probs = batch_probs_for_all_detectors(
        merged["target_text"].astype(str).tolist(),
        project_root=PROJECT_ROOT,
        split=SPLIT,
    )

    for det, probs in detector_probs.items():
        merged[f"prob_{det}"] = probs
        merged[f"flip_{det}"] = [
            (p >= 0.5) if d == "n2s" else (p < 0.5)
            for p, d in zip(probs, merged["direction"])
        ]

    if primary is not None:
        merged["flip_primary"] = merged[f"flip_{primary}"]
    else:
        merged["flip_primary"] = np.nan

    return merged

a_eval = enrich_outputs(a.copy())
b_eval = enrich_outputs(b.copy())
c_no_eval = enrich_outputs(c_no.copy())
c_yes_eval = enrich_outputs(c_yes.copy())

systems = {
    "System A (Template)": a_eval,
    "System B (Enc-Dec)": b_eval,
    "System C (No Retrieval)": c_no_eval,
    "System C (With Retrieval)": c_yes_eval,
}

def summarize_overall(df):
    out = {
        "Anchor preservation (%)": 100.0 * float(df["anchors_preserved"].mean()),
        "Semantic similarity (mean)": float(df["semantic_similarity"].mean()),
    }
    if "flip_primary" in df.columns:
        out[f"Flip rate — primary ({primary}) (%)"] = 100.0 * float(df["flip_primary"].mean())
    return out

overall = pd.DataFrame({name: summarize_overall(df) for name, df in systems.items()})
display(overall)


Available detectors: {'logreg': '/content/CS4248_Project/artifacts/classifiers/logreg/topic_hard', 'rnn': '/content/CS4248_Project/artifacts/classifiers/rnn/topic_hard', 'transformer': '/content/CS4248_Project/artifacts/classifiers/transformer/topic_hard'}
Primary detector: transformer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,System A (Template),System B (Enc-Dec),System C (No Retrieval),System C (With Retrieval)
Anchor preservation (%),100.000000,86.000000,100.000000,99.000000
Semantic similarity (mean),0.985573,0.891663,0.901559,0.889916
Flip rate — primary (transformer) (%),31.114226,28.000000,52.000000,54.000000


Build the per-detector flip table

In [ ]:
detector_rows = []
for det in [d for d in ("transformer", "rnn", "logreg") if f"flip_{d}" in next(iter(systems.values())).columns]:
    detector_rows.append({
        "detector": det,
        **{name: 100.0 * float(df[f"flip_{det}"].mean()) for name, df in systems.items()}
    })

detector_df = pd.DataFrame(detector_rows).set_index("detector")
display(detector_df)


,System A (Template),System B (Enc-Dec),System C (No Retrieval),System C (With Retrieval)
detector,,,,
transformer,31.114226,28.0,52.0,54.0
rnn,33.181500,44.0,75.0,77.0
logreg,33.181500,42.0,83.0,88.0


Build the by-direction tables

In [ ]:
for direction, label in [("n2s", "Neutral -> Sarcastic"), ("s2n", "Sarcastic -> Neutral")]:
    print(f"\n=== {label} ===")
    subset_systems = {name: df[df["direction"] == direction].copy() for name, df in systems.items()}

    direction_summary = pd.DataFrame({
        name: {
            "Anchor preservation (%)": 100.0 * float(sdf["anchors_preserved"].mean()),
            "Semantic similarity (mean)": float(sdf["semantic_similarity"].mean()),
            f"Flip rate — primary ({primary}) (%)": 100.0 * float(sdf["flip_primary"].mean()),
        }
        for name, sdf in subset_systems.items()
    })
    display(direction_summary)

    direction_det_rows = []
    for det in [d for d in ("transformer", "rnn", "logreg") if f"flip_{d}" in next(iter(subset_systems.values())).columns]:
        direction_det_rows.append({
            "detector": det,
            **{name: 100.0 * float(sdf[f"flip_{det}"].mean()) for name, sdf in subset_systems.items()}
        })
    direction_det_df = pd.DataFrame(direction_det_rows).set_index("detector")
    display(direction_det_df)



=== Neutral -> Sarcastic ===


,System A (Template),System B (Enc-Dec),System C (No Retrieval),System C (With Retrieval)
Anchor preservation (%),100.000000,82.812500,100.000000,100.000000
Semantic similarity (mean),0.973667,0.899933,0.916399,0.897641
Flip rate — primary (transformer) (%),46.757458,25.000000,40.625000,40.625000


,System A (Template),System B (Enc-Dec),System C (No Retrieval),System C (With Retrieval)
detector,,,,
transformer,46.757458,25.0000,40.625,40.6250
rnn,42.607004,48.4375,75.000,76.5625
logreg,46.303502,45.3125,84.375,87.5000



=== Sarcastic -> Neutral ===


,System A (Template),System B (Enc-Dec),System C (No Retrieval),System C (With Retrieval)
Anchor preservation (%),100.000000,91.666667,100.000000,97.222222
Semantic similarity (mean),0.999567,0.876961,0.875176,0.876183
Flip rate — primary (transformer) (%),12.728659,33.333333,72.222222,77.777778


,System A (Template),System B (Enc-Dec),System C (No Retrieval),System C (With Retrieval)
detector,,,,
transformer,12.728659,33.333333,72.222222,77.777778
rnn,22.103659,36.111111,75.000000,77.777778
logreg,17.759146,36.111111,80.555556,88.888889


Manually compare 100 test headlines outputs

In [ ]:
import json
import pandas as pd

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

a = pd.DataFrame(read_jsonl("artifacts/system_a/test_outputs.jsonl"))
b = pd.DataFrame(read_jsonl("artifacts/system_b/topic_hard/test_outputs.jsonl"))
c_no = pd.DataFrame(read_jsonl("results/system_c_outputs_topic_hard_test_no_retrieval.jsonl"))
c_yes = pd.DataFrame(read_jsonl("results/system_c_outputs_topic_hard_test.jsonl"))

a = a.rename(columns={"target_text": "system_a_output"})
b = b.rename(columns={"target_text": "system_b_output"})
c_no = c_no.rename(columns={"target_text": "system_c_no_retrieval"})
c_yes = c_yes.rename(columns={"target_text": "system_c_with_retrieval"})

compare = a[["id", "direction", "source_text", "system_a_output"]].copy()
compare = compare.merge(
    b[["id", "system_b_output"]],
    on="id",
    how="inner"
)
compare = compare.merge(
    c_no[["id", "system_c_no_retrieval"]],
    on="id",
    how="inner"
)
compare = compare.merge(
    c_yes[["id", "system_c_with_retrieval"]],
    on="id",
    how="inner"
)

display(compare.head(100))


,id,direction,source_text,system_a_output,system_b_output,system_c_no_retrieval,system_c_with_retrieval
0,sar_000006,n2s,my white inheritance,Experts confirm my white inheritance,"my white inheritance, apparently because this ...",white inheritance screams.,"I'll just give a tiny kiss for you, honey."
1,sar_000008,s2n,richard branson's global-warming donation near...,richard branson's global-warming donation near...,richard branson's global-warming donation cost...,richard branson's global-warming donation near...,Richard Branson's global warming donation near...
2,sar_000036,n2s,beyoncé sculpted in cheese is strangely alluring,Experts confirm beyoncé sculpted in cheese is ...,"as if sculpture wasn't weird enough, beyoncé s...",Beyonce makes sculpted cheeses as part of thei...,Beyonce's cheese cheese.
3,sar_000037,s2n,expansive obama state of the union speech to t...,expansive obama state of the union speech to t...,"address begins on robert altman films ahead,.",expansive obama state of the union speech to t...,expansive obama state of the union speech to t...
4,sar_000039,n2s,police say woman made up story of attack by tw...,Nation relieved as police say woman made up st...,police say woman elaborated on murder claims a...,A woman has been reported assaulted by two men...,Sources confirm police say woman made up story...
...,...,...,...,...,...,...,...
95,sar_001123,s2n,employees from other department announce plan ...,employees from other department announce plan ...,workers of other department announce news abou...,New staff members announce upcoming shift and ...,idb: wiki: The US Government’s response to a t...
96,sar_001147,n2s,dear critical white scholar and colleague:,dear critical white scholar and colleague: in ...,"dear critical white scholar and colleague, eve...",'Arsenal of White Scholars' :.,He’s a vicious critic of every political think...
97,sar_001150,n2s,9 brands with sexy spokesmodels over 50,9 brands with sexy spokesmodels over 50 in stu...,9 brands with sexy spokesmodels over 50 over t...,Sources confirm 9 brands with sexy spokesmodel...,Sources confirm 9 brands with sexy spokesmodel...
98,sar_001167,n2s,the real reason silicon valley is the world's ...,Experts confirm the real reason silicon valley...,its elusive tourist-site now explains why sili...,'Silicon Valley' s elusive tourist attraction.,Silicon Valley's lack of tourists will make it...


In [ ]:
# Save tables
save_table(overall, "overall_metrics.csv", index=True)
save_top100(compare, "system_outputs_compare_top100.csv")

# Copy generated jsonl files into Drive
copy_if_exists("artifacts/system_a/test_outputs.jsonl")
copy_if_exists("artifacts/system_b/topic_hard/test_outputs.jsonl")
copy_if_exists("results/system_c_outputs_topic_hard_test_no_retrieval.jsonl")
copy_if_exists("results/system_c_outputs_topic_hard_test.jsonl")

# Save a full snapshot of the repo
backup_repo_zip(tag="final")

Saved table -> /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/tables/overall_metrics.csv
Saved top100 -> /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/tables/system_outputs_compare_top100.csv
Copied -> /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/generated_files/artifacts/system_a/test_outputs.jsonl
Copied -> /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/generated_files/artifacts/system_b/topic_hard/test_outputs.jsonl
Copied -> /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/generated_files/results/system_c_outputs_topic_hard_test_no_retrieval.jsonl
Copied -> /content/drive/MyDrive/CS4248_colab/preprocessing/outputs/generated_files/results/system_c_outputs_topic_hard_test.jsonl
Repo snapshot saved -> /content/drive/MyDrive/CS4248_colab/preprocessing/repo_snapshots/CS4248_Project_final_20260414_162502.zip


PosixPath('/content/drive/MyDrive/CS4248_colab/preprocessing/repo_snapshots/CS4248_Project_final_20260414_162502.zip')